# 01 · Data preparation, EDA, and features

Run this notebook from top to bottom. It combines the Chennai GTFS audit, scheduled route/hour supply analysis, and route feature preparation. Inputs are read from `dataset/raw/chennai_gtfs`; generated CSVs are written to `dataset/processed`.

The feed contains scheduled service, not observed passenger counts. The synthetic scenario is handled in notebook 02 and must remain labeled as synthetic.

## Section 1 · GTFS data audit

In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
from src.gtfs import resolve_gtfs_dir, load_gtfs

GTFS_DIR = resolve_gtfs_dir(PROJECT_ROOT)
feed = load_gtfs(GTFS_DIR)
print(f"Feed folder: {GTFS_DIR}")


### 1. Table sizes and columns

In [ ]:
for name, df in feed.items():
    print(f"{name:12s} rows={len(df):,} columns={len(df.columns)}")
    display(df.head(3))

### 2. Required fields, missing values, and key uniqueness

In [ ]:
key_cols = {"agency": ["agency_id"], "calendar": ["service_id"], "routes": ["route_id"],
            "stops": ["stop_id"], "trips": ["trip_id"]}
for name, cols in key_cols.items():
    df = feed[name]
    missing_cols = sorted(set(cols) - set(df.columns))
    duplicate_keys = int(df.duplicated(cols).sum()) if not missing_cols else None
    print(f"{name}: missing key columns={missing_cols}; duplicate keys={duplicate_keys}")

for name, df in feed.items():
    blank_cells = int(df.replace("", pd.NA).isna().sum().sum())
    print(f"{name}: blank cells={blank_cells:,}")

### 3. Foreign-key checks and service date coverage

In [ ]:
route_ids = set(feed["routes"]["route_id"])
trip_route_ids = set(feed["trips"]["route_id"])
trip_ids = set(feed["trips"]["trip_id"])
stop_ids = set(feed["stops"]["stop_id"])
service_ids = set(feed["calendar"]["service_id"])

checks = {
    "trip route IDs absent from routes.txt": len(trip_route_ids - route_ids),
    "trip service IDs absent from calendar.txt": len(set(feed["trips"]["service_id"]) - service_ids),
    "stop_times trip IDs absent from trips.txt": len(set(feed["stop_times"]["trip_id"]) - trip_ids),
    "stop_times stop IDs absent from stops.txt": len(set(feed["stop_times"]["stop_id"]) - stop_ids),
}
for label, count in checks.items():
    print(f"{label}: {count:,}")

cal = feed["calendar"]
print("Calendar date span:", cal["start_date"].min().date(), "to", cal["end_date"].max().date())
print("Agency names:", feed["agency"]["agency_name"].unique().tolist())
print("Route type counts:")
display(feed["routes"]["route_type"].value_counts(dropna=False).rename_axis("route_type").to_frame("routes"))

### 4. What this audit tells us

- `trips.txt` and `stop_times.txt` measure **scheduled supply**, not passengers.
- `calendar.txt` says when a service is scheduled to operate. The feed also has no `calendar_dates.txt`, so this first version cannot model service exceptions or cancellations.
- For actual route-level ridership, seek MTC ticketing / passenger-count data or collect a small survey. If the project must proceed without it, the synthetic-scenario section below creates a **synthetic scenario** calibrated to an aggregate daily total. That is a planning assumption, not ground truth.

Save the audit findings in your report, including any nonzero key-check results.

## Section 2 · Scheduled route/hour supply

In [ ]:
from src.gtfs import active_service_ids, first_trip_departures

# Edit this to a date covered by calendar.txt. This feed covers 2024-05-01 through 2030-05-01.
SERVICE_DATE = "2026-10-26"  # Monday
active_ids = active_service_ids(feed["calendar"], SERVICE_DATE)
print(f"Active service IDs on {SERVICE_DATE}: {len(active_ids)}")
if not active_ids:
    raise ValueError("No calendar service is active on SERVICE_DATE; choose a date within the feed coverage.")

### 1. Count scheduled trips by route and departure hour

In [ ]:
departures = first_trip_departures(feed["stop_times"])
trips = feed["trips"].copy()
trips["service_id"] = trips["service_id"].astype(str)
trips = trips[trips["service_id"].isin(active_ids)].merge(departures, on="trip_id", how="left", validate="one_to_one")
trips = trips.merge(feed["routes"], on="route_id", how="left", validate="many_to_one")
trips = trips.dropna(subset=["departure_hour"])
trips["departure_hour"] = trips["departure_hour"].astype(int)

route_hour_supply = (trips.groupby(["route_id", "route_short_name", "route_long_name", "departure_hour"], dropna=False)
                     .size().rename("scheduled_trip_count").reset_index())
print(f"Scheduled trips counted: {len(trips):,}")
display(route_hour_supply.head(10))

### 2. Route summary and supply chart

In [ ]:
route_summary = (route_hour_supply.groupby(["route_id", "route_short_name", "route_long_name"], dropna=False)
                 .agg(scheduled_trips=("scheduled_trip_count", "sum"),
                      operating_departure_hours=("departure_hour", "nunique"))
                 .reset_index().sort_values("scheduled_trips", ascending=False))
display(route_summary.head(20))

hourly_supply = route_hour_supply.groupby("departure_hour")["scheduled_trip_count"].sum().reindex(range(24), fill_value=0)
ax = hourly_supply.plot(kind="bar", figsize=(12, 4), color="#247ba0", title=f"Scheduled trip departures by hour · {SERVICE_DATE}")
ax.set_xlabel("Departure hour (local feed time)")
ax.set_ylabel("GTFS scheduled trips")
ax.grid(axis="y", alpha=.25)


### 3. Export schedule features

The CSV is generated locally and is ignored by Git. It contains schedule-derived features only.

In [ ]:
output_dir = PROJECT_ROOT / "dataset" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)
supply_path = output_dir / f"chennai_route_hour_supply_{SERVICE_DATE}.csv"
route_hour_supply.to_csv(supply_path, index=False)
print(f"Wrote {supply_path}")

### Interpretation

A route with more scheduled trips has more service supply in the feed. It does not follow that it has more riders or is overcrowded. Use the supply table as a baseline for comparing a later ridership dataset or as the allocation basis for the explicitly synthetic exercise in the synthetic-scenario section.

## Section 3 · Route EDA and feature preparation

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
from src.gtfs import resolve_gtfs_dir

GTFS_DIR = resolve_gtfs_dir(PROJECT_ROOT)
PROCESSED_DIR = PROJECT_ROOT / "dataset" / "processed"
supply_files = sorted(PROCESSED_DIR.glob("chennai_route_hour_supply_*.csv"))
if not supply_files:
    raise FileNotFoundError("Run the scheduled-supply section in this notebook first; no route-hour supply CSV was found in dataset/processed.")
supply_path = supply_files[-1]
SERVICE_DATE = supply_path.stem.rsplit("_", 1)[-1]
ridership_path = PROCESSED_DIR / f"synthetic_route_hour_ridership_{SERVICE_DATE}.csv"
if not ridership_path.is_file():
    raise FileNotFoundError(f"Run the synthetic-scenario section in notebook 02 for {SERVICE_DATE}; missing {ridership_path.name}.")

supply = pd.read_csv(supply_path, dtype={"route_id": str})
synthetic = pd.read_csv(ridership_path, dtype={"route_id": str})
print(f"Service date: {SERVICE_DATE}")
print(f"Input folder: {GTFS_DIR}")
print(f"Supply rows: {len(supply):,}; synthetic rows: {len(synthetic):,}")
print(f"Synthetic label(s): {synthetic['data_status'].unique().tolist()}")

### 1. Verify that the two exports describe the same route-hour rows

In [ ]:
keys = ["route_id", "departure_hour"]
if supply.duplicated(keys).any() or synthetic.duplicated(keys).any():
    raise ValueError("Duplicate route-hour keys found; resolve before analysis.")

scenario_cols = keys + ["synthetic_ridership_estimate", "scenario_multiplier", "data_status", "reference_daily_total", "reference_source"]
route_hour = supply.merge(synthetic[scenario_cols], on=keys, how="outer", validate="one_to_one", indicator=True)
if not route_hour["_merge"].eq("both").all():
    raise ValueError("Supply and synthetic files do not contain the same route-hour rows.")
route_hour = route_hour.drop(columns="_merge")

schedule_total = int(route_hour["scheduled_trip_count"].sum())
scenario_total = int(route_hour["synthetic_ridership_estimate"].sum())
reference_total = int(route_hour["reference_daily_total"].iloc[0])
print(f"Scheduled trip departures: {schedule_total:,}")
print(f"Synthetic scenario total: {scenario_total:,}")
print(f"Aggregate calibration total: {reference_total:,}")
print("Synthetic total matches calibration:", scenario_total == reference_total)
print("Provenance:", route_hour["reference_source"].iloc[0])
display(route_hour.head())

### 2. Explore hourly service supply and the synthetic hourly profile

In [ ]:
PEAK_HOURS = {7, 8, 9, 17, 18, 19}
hour_summary = (route_hour.groupby("departure_hour")
                .agg(scheduled_trip_departures=("scheduled_trip_count", "sum"),
                     synthetic_estimate=("synthetic_ridership_estimate", "sum"))
                .reindex(range(24), fill_value=0))
display(hour_summary)

ax = hour_summary.plot(kind="bar", secondary_y="synthetic_estimate", figsize=(13, 5),
                       color={"scheduled_trip_departures":"#247ba0", "synthetic_estimate":"#f18f01"},
                       title=f"Chennai scheduled supply and synthetic allocation by hour · {SERVICE_DATE}")
ax.set_xlabel("Departure hour (feed local time)")
ax.set_ylabel("Scheduled trip departures")
ax.right_ax.set_ylabel("Synthetic passenger estimate")
ax.grid(axis="y", alpha=.2)
plt.tight_layout()

route_supply = (route_hour.groupby(["route_id", "route_short_name", "route_long_name"], dropna=False)
                .agg(scheduled_trips=("scheduled_trip_count", "sum"),
                     operating_hours=("departure_hour", "nunique"),
                     first_departure_hour=("departure_hour", "min"),
                     last_departure_hour=("departure_hour", "max"),
                     synthetic_daily_estimate=("synthetic_ridership_estimate", "sum"))
                .sort_values("scheduled_trips", ascending=False).reset_index())
display(route_supply.head(20))

### 3. Add route structure features from GTFS

In [ ]:
# Read only the GTFS fields needed for these features to limit memory use.
trips = pd.read_csv(GTFS_DIR / "trips.txt", dtype=str, usecols=["trip_id", "route_id"])
stop_times = pd.read_csv(GTFS_DIR / "stop_times.txt", dtype=str,
                         usecols=["trip_id", "stop_id", "stop_sequence", "arrival_time", "departure_time"])
stop_times["stop_sequence"] = pd.to_numeric(stop_times["stop_sequence"], errors="coerce")
trip_route = trips.drop_duplicates("trip_id")
trip_stops = stop_times[["trip_id", "stop_id"]].drop_duplicates().merge(
    trip_route, on="trip_id", how="inner", validate="many_to_one")

# Number of distinct stops served by a route and median stop count for one scheduled trip.
route_stop_count = trip_stops.drop_duplicates(["route_id", "stop_id"]).groupby("route_id").size().rename("distinct_stop_count")
trip_stop_count = (trip_stops.groupby(["route_id", "trip_id"])["stop_id"].nunique()
                   .groupby("route_id").median().rename("median_stops_per_trip"))

# Approximate scheduled journey duration from first departure to last arrival.
def gtfs_time_to_minutes(values):
    parts = values.astype(str).str.extract(r"^(?P<hour>\d+):(?P<minute>\d{2}):(?P<second>\d{2})$")
    return (pd.to_numeric(parts["hour"], errors="coerce") * 60
            + pd.to_numeric(parts["minute"], errors="coerce")
            + pd.to_numeric(parts["second"], errors="coerce") / 60)

ordered = stop_times.sort_values(["trip_id", "stop_sequence"], kind="stable")
first_stops = ordered.drop_duplicates("trip_id", keep="first")[["trip_id", "departure_time"]]
last_stops = ordered.drop_duplicates("trip_id", keep="last")[["trip_id", "arrival_time"]]
trip_duration = first_stops.merge(last_stops, on="trip_id", validate="one_to_one").merge(
    trip_route, on="trip_id", validate="one_to_one")
trip_duration["duration_minutes"] = gtfs_time_to_minutes(trip_duration["arrival_time"]) - gtfs_time_to_minutes(trip_duration["departure_time"])
# GTFS permits after-midnight times above 24:00; the fallback handles feeds that wrap at midnight.
trip_duration.loc[trip_duration["duration_minutes"] < 0, "duration_minutes"] += 24 * 60
median_duration = trip_duration.groupby("route_id")["duration_minutes"].median().rename("median_trip_duration_min")

route_structure = pd.concat([route_stop_count, trip_stop_count, median_duration], axis=1).reset_index()
print(f"GTFS routes with stop-time structure: {route_structure['route_id'].nunique():,}")
display(route_structure.head())

### 4. Build and save the route-level feature table

In [ ]:
base = (route_hour.groupby(["route_id", "route_short_name", "route_long_name"], dropna=False)
        .agg(scheduled_trips=("scheduled_trip_count", "sum"),
             scheduled_hours=("departure_hour", "nunique"),
             first_departure_hour=("departure_hour", "min"),
             last_departure_hour=("departure_hour", "max"),
             synthetic_daily_estimate=("synthetic_ridership_estimate", "sum"),
             synthetic_peak_estimate=("synthetic_ridership_estimate", lambda s: int(s[route_hour.loc[s.index, "departure_hour"].isin(PEAK_HOURS)].sum())))
        .reset_index())
base["service_span_hours"] = base["last_departure_hour"] - base["first_departure_hour"] + 1
base["peak_trip_share"] = (route_hour[route_hour["departure_hour"].isin(PEAK_HOURS)]
                            .groupby("route_id")["scheduled_trip_count"].sum()
                            .div(base.set_index("route_id")["scheduled_trips"]).reindex(base["route_id"]).to_numpy())
base["synthetic_estimate_per_trip"] = base["synthetic_daily_estimate"] / base["scheduled_trips"]
base["synthetic_peak_estimate_share"] = base["synthetic_peak_estimate"] / base["synthetic_daily_estimate"]
base = base.drop(columns="synthetic_peak_estimate")

hourly = route_hour.pivot_table(index="route_id", columns="departure_hour", values="scheduled_trip_count", aggfunc="sum", fill_value=0)
hourly = hourly.reindex(columns=range(24), fill_value=0)
hourly = hourly.div(hourly.sum(axis=1), axis=0).fillna(0)
hourly.columns = [f"trip_share_h{int(h):02d}" for h in hourly.columns]

features = (base.merge(route_structure, on="route_id", how="left", validate="one_to_one")
            .merge(hourly.reset_index(), on="route_id", how="left", validate="one_to_one"))
features["service_date"] = SERVICE_DATE
features["ridership_data_status"] = "synthetic_scenario_not_observed"

output_path = PROCESSED_DIR / f"route_features_{SERVICE_DATE}.csv"
features.to_csv(output_path, index=False)
print(f"Saved {len(features):,} route feature rows to {output_path}")
print(f"Feature columns: {len(features.columns)}")
display(features.sort_values("scheduled_trips", ascending=False).head(20))

### 5. Candidate features for later clustering

The schedule-based feature set below can be standardized and clustered as an exploratory service-pattern analysis. The columns beginning `synthetic_` are excluded. They are useful for scenario charts, but they are derived from assumptions and should not be treated as measured passenger demand.

No shapes file is included, so route distance and actual in-vehicle passenger load are unavailable. The median trip duration is a scheduled-time proxy, not a traffic-observed travel time.

In [ ]:
profile_columns = [f"trip_share_h{hour:02d}" for hour in range(24)]
SCHEDULE_FEATURES = ["scheduled_trips", "scheduled_hours", "service_span_hours",
                     "peak_trip_share", "distinct_stop_count", "median_stops_per_trip",
                     "median_trip_duration_min"] + profile_columns
missing_features = [name for name in SCHEDULE_FEATURES if name not in features.columns]
if missing_features:
    raise ValueError(f"Expected schedule feature(s) missing: {missing_features}")
print(f"Schedule-only clustering features prepared: {len(SCHEDULE_FEATURES)}")
display(features[SCHEDULE_FEATURES].describe().T)

### Interpretation and next project step

- This notebook checks that supply and synthetic exports align and preserves the exact aggregate calibration total.
- The route feature file is now ready for **exploratory clustering of scheduled service patterns**. It is not a passenger-demand training set.
- Next, compare a simple clustering baseline (for example, scaled KMeans) using only `SCHEDULE_FEATURES`, inspect whether clusters are understandable, and keep `synthetic_*` fields out of the clustering inputs.
- Before making overcrowding or ridership claims, obtain route-level observed counts or conduct a documented passenger survey.